# Slack Approval Gate — human-in-the-loop spend

Agent creates a **payment intent** (authorize only). Human approves in the next cell (Slack stand-in), then intent is **confirmed**.

**Prereq:** `pip install -e "./python[dev]" jupyter`

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from _notebook_utils import FAKE_RECIPIENT, enable_mock_router

from algopay import AlgoPay
from algopay.core.types import Network

AMOUNT = "0.25"
PURPOSE = "Premium data API - Q1 report"

client = AlgoPay(network=Network.ALGORAND_TESTNET)
enable_mock_router(client)

ws = client.wallet.create_wallet_set("slack-gate")
wallet = client.wallet.create_wallet(ws.id)
await client.add_single_tx_guard(wallet.id, max_amount="1.0")

In [ ]:
print("Agent requests payment (authorize only — no spend yet)...")
intent = await client.create_payment_intent(
    wallet.id, FAKE_RECIPIENT, AMOUNT, purpose=PURPOSE
)
print(f"Intent id: {intent.id}")
print(f"Status: {intent.status.value}")
print(f"Amount: {AMOUNT} USDC")
print(f"Purpose: {PURPOSE}")

## Human approval (Slack stand-in)

Set `HUMAN_APPROVED` to `True` or `False` in the cell below, then run it.

In [ ]:
HUMAN_APPROVED = True  # flip to False to demo rejection

if HUMAN_APPROVED:
    print("[Slack #agent-spend] Approved by human operator.")
    result = await client.confirm_payment_intent(intent.id)
    if result.success:
        print(f"Payment completed: {result.blockchain_tx}")
    else:
        print(f"Payment failed: {result.error}")
else:
    print("[Slack #agent-spend] Rejected — no funds moved.")

In [ ]:
from _notebook_utils import display_ledger

entries = await client.ledger.query(wallet_id=wallet.id, limit=10)
display_ledger(entries)